# Track B: Deep Ensemble (CNN) on N-CMAPSS DS02 raw data

Counterpart to Track A (NGBoost on cycle-aggregated features, see the local repo). Here: a 1D CNN with a Gaussian NLL head, as a deep ensemble (5 members) for uncertainty quantification, trained on sliding-window sequences of the raw 1 Hz sensor data.

**Calibration methodology deliberately differs from Track A:** there, only 6 units were available, hence nested cross-conformal. Here there are hundreds of thousands of windows, so a simple split-conformal approach with one fully held-out calibration unit is enough — no nested CV needed, and considerably cheaper to compute.

In [ ]:
import glob
import os
import shutil
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


def find_one(pattern):
    matches = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"No match for {pattern!r} under /kaggle/input. "
            f"Full tree: {list(os.walk('/kaggle/input'))[:20]}"
        )
    return matches[0]


# Kaggle's input mount layout has changed over time (flat vs. nested under
# an extra 'datasets/' directory); searching recursively avoids hardcoding
# an assumption that breaks on the next platform change.
data_py_path = find_one("data.py")
SRC_INPUT = os.path.dirname(data_py_path)
print("found turbofan_rul source at:", SRC_INPUT)

PKG_ROOT = "/kaggle/working/pkg"
PKG_DIR = os.path.join(PKG_ROOT, "turbofan_rul")
if not os.path.exists(PKG_DIR):
    os.makedirs(PKG_ROOT, exist_ok=True)
    shutil.copytree(SRC_INPUT, PKG_DIR)
sys.path.insert(0, PKG_ROOT)

from turbofan_rul.data import load_ncmapss_h5
from turbofan_rul.features import feature_columns
from turbofan_rul.sequences import make_windows, standardize_features
from turbofan_rul.track_b import train_ensemble, predict_ensemble, pick_device
from turbofan_rul.calibration import nonconformity_scores, conformal_quantile, conformal_interval
from turbofan_rul.evaluate import rmse, nasa_score, coverage, clip_rul

DEVICE = pick_device()
print("device:", DEVICE, " gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

DATA_PATH = find_one("N-CMAPSS_DS02.h5")
print("found DS02 at:", DATA_PATH)
df_dev_raw = load_ncmapss_h5(DATA_PATH, split="dev")
df_test_raw = load_ncmapss_h5(DATA_PATH, split="test")
feat_cols = feature_columns(df_dev_raw)
print("dev shape:", df_dev_raw.shape, " test shape:", df_test_raw.shape, " n_features:", len(feat_cols))


## Train / calibration / test split

One dev unit is held out entirely for conformal calibration (no leakage between training and calibration); the remaining 5 units train the ensemble. Test remains, as in Track A, the official DS02 test set (units 11/14/15, all three flight classes).

In [ ]:
dev_units = sorted(df_dev_raw["a_unit"].unique())
CALIBRATION_UNIT = dev_units[0]
train_units = [u for u in dev_units if u != CALIBRATION_UNIT]
print("train units:", train_units, " calibration unit:", CALIBRATION_UNIT)

WINDOW = 50
STRIDE = 1
SUBSAMPLE = 10

train_df = df_dev_raw[df_dev_raw["a_unit"].isin(train_units)]
calib_df = df_dev_raw[df_dev_raw["a_unit"] == CALIBRATION_UNIT]

t0 = time.time()
X_train, y_train, _, _ = make_windows(train_df, feat_cols, window=WINDOW, stride=STRIDE, subsample=SUBSAMPLE)
X_calib, y_calib, _, _ = make_windows(calib_df, feat_cols, window=WINDOW, stride=STRIDE, subsample=SUBSAMPLE)
X_test, y_test, units_test, cycles_test = make_windows(df_test_raw, feat_cols, window=WINDOW, stride=STRIDE, subsample=SUBSAMPLE)
print("windowing took", round(time.time() - t0, 1), "s")
print("train:", X_train.shape, " calib:", X_calib.shape, " test:", X_test.shape)

# Raw sensor channels span very different physical scales (temperatures in
# the hundreds, speeds in the thousands RPM) -- standardize using train-set
# stats only, then reuse those stats for calib/test (no refitting, avoids
# leakage). Without this the CNN's loss goes to NaN within the first epoch.
X_train, feat_mean, feat_std = standardize_features(X_train)
X_calib, _, _ = standardize_features(X_calib, mean=feat_mean, std=feat_std)
X_test, _, _ = standardize_features(X_test, mean=feat_mean, std=feat_std)
print("feature means (first 5):", feat_mean[:5])
print("feature stds (first 5): ", feat_std[:5])

## Train the deep ensemble (5 members, CNN + Gaussian NLL head)

In [ ]:
t0 = time.time()
models = train_ensemble(
    X_train, y_train,
    n_members=5, n_epochs=15, batch_size=256, lr=1e-3, hidden=32,
    device=DEVICE, verbose=True,
)
print("training took", round((time.time() - t0) / 60, 1), "min")

## Calibration (split conformal on the held-out unit)

In [ ]:
mean_calib, scale_calib = predict_ensemble(models, X_calib, device=DEVICE)
scores = nonconformity_scores(y_calib, mean_calib, scale_calib)
q_hat = conformal_quantile(scores, target_coverage=0.9)
print("q_hat:", round(q_hat, 3))

## Evaluation on the official test set (units 11/14/15, all flight classes)

Naive intervals use the ensemble directly (mean +/- 1.645*scale, 90% normal quantile); conformal uses `q_hat` from calibration — same comparison as in the Track A notebook.

In [ ]:
from scipy.stats import norm

mean_test, scale_test = predict_ensemble(models, X_test, device=DEVICE)
z = norm.ppf(0.95)  # 90% central interval
lower_naive, upper_naive = mean_test - z * scale_test, mean_test + z * scale_test
lower_conf, upper_conf = conformal_interval(mean_test, scale_test, q_hat)

# RUL cannot be negative -- near end-of-life the raw interval math routinely
# dips below zero (real incident: -11.7 "cycles remaining" in Track A before
# this fix). Clip both intervals and the point prediction before reporting.
mean_test, lower_naive, upper_naive = clip_rul(mean_test, lower_naive, upper_naive)
mean_test, lower_conf, upper_conf = clip_rul(mean_test, lower_conf, upper_conf)

test_rows = []
for unit in sorted(np.unique(units_test)):
    fc = df_test_raw.loc[df_test_raw["a_unit"] == unit, "a_Fc"].iloc[0]
    mask = units_test == unit
    test_rows.append({
        "unit": unit,
        "flight_class": fc,
        "n_windows": int(mask.sum()),
        "rmse": rmse(y_test[mask], mean_test[mask]),
        "nasa_score": nasa_score(y_test[mask], mean_test[mask]),
        "coverage_90_naive": coverage(y_test[mask], lower_naive[mask], upper_naive[mask]),
        "coverage_90_conformal": coverage(y_test[mask], lower_conf[mask], upper_conf[mask]),
    })

test_results = pd.DataFrame(test_rows)
test_results

In [ ]:
print("Overall (all test units, Track B):")
print("RMSE:", rmse(y_test, mean_test))
print("NASA score:", nasa_score(y_test, mean_test))
print("Coverage (90%) naive:    ", coverage(y_test, lower_naive, upper_naive))
print("Coverage (90%) conformal:", coverage(y_test, lower_conf, upper_conf))

## Degradation trajectory per test unit

In [ ]:
test_units = sorted(np.unique(units_test))
fig, axes = plt.subplots(1, len(test_units), figsize=(5 * len(test_units), 4), sharey=True)

for ax, unit in zip(axes, test_units):
    mask = units_test == unit
    fc = df_test_raw.loc[df_test_raw["a_unit"] == unit, "a_Fc"].iloc[0]
    order = np.argsort(cycles_test[mask])
    c = cycles_test[mask][order]

    ax.plot(c, y_test[mask][order], "k-", label="true RUL")
    ax.plot(c, mean_test[mask][order], "b-", label="prediction (mean)")
    ax.fill_between(c, lower_conf[mask][order], upper_conf[mask][order], color="blue", alpha=0.2, label="conformal (90%)")
    ax.set_title(f"Unit {int(unit)} (Flight Class {int(fc)})")
    ax.set_xlabel("Cycle (window end)")

axes[0].set_ylabel("RUL")
axes[0].legend()
fig.tight_layout()
fig.savefig("/kaggle/working/track_b_degradation_curves.png", dpi=120)

## Export results

A CSV with the test-set metrics ends up under `/kaggle/working/` — downloadable as kernel output and to be copied back into the local repo under `outputs/`, for the comparison with Track A in `docs/results.md`.

In [ ]:
test_results.to_csv("/kaggle/working/track_b_test_results.csv", index=False)
summary = pd.DataFrame([{
    "rmse": rmse(y_test, mean_test),
    "nasa_score": nasa_score(y_test, mean_test),
    "coverage_90_naive": coverage(y_test, lower_naive, upper_naive),
    "coverage_90_conformal": coverage(y_test, lower_conf, upper_conf),
    "q_hat": q_hat,
    "n_train_windows": len(X_train),
    "n_calib_windows": len(X_calib),
    "n_test_windows": len(X_test),
}])
summary.to_csv("/kaggle/working/track_b_summary.csv", index=False)
summary